# Act 1 — The Well-Architected Framework

Thirteen notebooks in, you have the AWS service map. This one ties the threads together. AWS's **Well-Architected Framework (WAF)** is the structured way the field talks about *"is this architecture good?"* — six pillars, each with design principles, best-practice questions, and a portal-based scoring tool. Every service you have already met fits under one or more of these pillars; the work in this act is composition rather than introduction.

The six pillars are not independent. Tightening security often costs more. Improving reliability usually trades against cost. Sustainability sometimes argues for the same choices as cost. The WAF review process exists precisely to make those trade-offs explicit per workload, so nobody silently optimises for one pillar at the expense of another.

## Operational Excellence

Operational Excellence is the pillar about *running the system well day to day*. Can you ship changes safely? Respond to incidents? Improve from what you learn?

The concrete checklist that emerges from the previous notebooks:

- **Infrastructure as code** for everything — CloudFormation, CDK, or Terraform. Manual console changes flagged and reverted.
- **Pipelines** — CodePipeline / CodeBuild / CodeDeploy or GitHub Actions, authenticating via OIDC federation (no long-lived IAM keys in CI).
- **Safe deployment** — blue/green for ECS and Lambda (CodeDeploy traffic shifting), canary for ALB, with automated rollback on alarm.
- **Observability** — CloudWatch Logs / Metrics / Alarms wired by default; CloudTrail enabled organisation-wide; AWS Config recording.
- **Runbooks** — Systems Manager Documents (Automation runbooks) for the common operational tasks; Incident Manager for response.
- **Improve from incidents** — post-incident reviews; SSM OpsCenter for action items; the same incident never twice.

The pillar's anti-pattern is the heroic operator who knows it all in their head. That works until they go on holiday.

## Security

Security is the pillar about *protecting data and identities against credible threats*. The non-negotiables:

- **Identity** — IAM roles, not users, for everything. Permission boundaries on developer roles. SSO via IAM Identity Center, not local IAM users.
- **Least privilege** — IAM Access Analyzer findings reviewed; policies sized to actual usage (CloudTrail-driven generation via `aws iam generate-service-last-accessed-details` and Access Analyzer policy generation).
- **Data encryption** — KMS CMKs for sensitive data; S3 default encryption; EBS default encryption flag enabled per Region; in-transit TLS everywhere.
- **Secrets** — Secrets Manager with rotation, or SSM Parameter Store SecureString; never plain-text secrets in code, AMIs, or Lambda env vars.
- **Network controls** — VPC with private subnets for compute, public subnets only where strictly needed; Security Groups deny-by-default; VPC Endpoints for AWS services; WAF and Shield on internet-facing endpoints.
- **Detection** — GuardDuty enabled everywhere, organisation-wide; Security Hub aggregating findings; Macie on data stores with PII.
- **Audit** — CloudTrail (org trail) to S3 with Object Lock; Config recording all resources; quarterly access reviews.

The customer-side of the Shared Responsibility Model (data and identity) is always yours; AWS protects the rest.

## Reliability

Reliability is the pillar about *meeting your uptime and recovery targets*. The checklist:

- **Distributed across AZs** for every production tier — ASGs span all AZs, RDS Multi-AZ, ElastiCache Multi-AZ, ALB cross-zone load balancing.
- **Stateless compute** — sessions in ElastiCache, files in S3, secrets in Secrets Manager. Local disk holds nothing of value.
- **Automatic recovery** — Auto Scaling self-heals failed instances; ALB health checks remove bad targets; EC2 Auto Recovery handles host failures; ECS service scheduler replaces failed tasks.
- **Service quotas managed** — request limit increases before you hit them, alarm on quota usage via Service Quotas console.
- **DR strategy chosen and tested** — Backup & Restore, Pilot Light, Warm Standby, or Multi-Site Active-Active per workload, with quarterly drills (notebook 13).
- **Backups with vault lock** — AWS Backup vaults in compliance mode, cross-Region and cross-account copies.
- **Resilient API design** — exponential backoff with jitter on every AWS SDK call, idempotency tokens on writes.

Reliability is *measured*, not asserted. If you can't quote your RTO/RPO per workload, you don't have a reliability story; you have a hope.

## Performance Efficiency

Performance Efficiency is the pillar about *delivering on workload performance targets without overspending*. The checklist:

- **Pick the right compute** — Lambda for sporadic event work, Fargate for steady containers, EC2 for sustained or custom workloads, Graviton (`m6g`, `c7g`) for ~20% better price/performance on most x86 workloads.
- **Pick the right data engine per access pattern** — RDS / Aurora for OLTP, DynamoDB for key-value at any scale, Redshift for warehouse, S3 + Athena for ad-hoc lake, OpenSearch for search, ElastiCache for hot.
- **Cache aggressively** — CloudFront at the edge, ElastiCache near the application, DAX in front of DynamoDB when latency demands.
- **Asynchronous patterns** — SQS, SNS, EventBridge, Step Functions decouple latency-sensitive paths from slow downstream work.
- **Auto-scaling rules with cooldowns** — never flap; scale-out aggressive, scale-in conservative.
- **Network optimisation** — placement groups for low-latency clusters; Enhanced Networking (ENA) and EFA for HPC; Global Accelerator for global latency.
- **Storage class matched to access** — gp3 over gp2 for EBS (cheaper and more performant); S3 Intelligent-Tiering for mixed access.

The lever that pays everyone back: **review the architecture every quarter** for newer instance generations and managed-service replacements that didn't exist when you designed it.

## Cost Optimization

Notebook 13 covered the cost levers in depth. The pillar's checklist, briefly:

- **Right-size weekly** — Compute Optimizer + Trusted Advisor + Cost Explorer.
- **Commit on the base** — Compute Savings Plans for 60–70% of fleet; RIs for stable RDS / ElastiCache.
- **Spot for the burst layer** — batch, CI, stateless workers.
- **Storage lifecycle** — S3 Intelligent-Tiering by default; EBS gp3 over gp2; delete unattached volumes.
- **Idle elimination** — auto-shutdown for dev/test; scheduled scaling for off-hours.
- **Tag governance** — `cost-center`, `environment`, `owner`, `application` on everything; Cost Explorer pivots by tag.
- **Anomaly detection on day one** — Cost Anomaly Detection catches the compromised-key cryptominer before the bill arrives.

The operating discipline matters more than any single optimisation: a team that reviews cost weekly will always beat a team that does an annual deep-dive.

## Sustainability

The newest pillar — added in 2021 — is about *minimising the environmental impact of running cloud workloads*. The lens is: less compute consumed, fewer resources idle, more efficient hardware.

Most of the checklist overlaps with Cost Optimization:

- **Right-size and decommission idle** — the same lever that saves money saves carbon.
- **Modern instance generations** — Graviton processors consume meaningfully less power per unit of work than older x86.
- **Serverless and managed services** — share underlying hardware, raise utilisation, reduce per-customer footprint.
- **Storage tiering** — Glacier Deep Archive at scale uses dramatically less energy than Standard for the same bytes.
- **Region choice** — AWS publishes carbon intensity per Region; some Regions are powered by significantly more renewable energy than others. For latency-flexible workloads, picking a low-carbon Region matters.
- **Customer Carbon Footprint Tool** — AWS reports your estimated emissions in the Billing console; track it like any other KPI.

The pillar reinforces the discipline: most cloud waste is also business waste.

# Act 2 — The tools — Well-Architected Tool and Trusted Advisor

The pillars give you the principles. Two AWS services turn them into operating practice you can score against:

- The **AWS Well-Architected Tool** walks you through the official questionnaire per workload and produces a list of high/medium/low risks against each pillar.
- **AWS Trusted Advisor** continuously scans your account for best-practice gaps across cost, performance, security, fault tolerance, and service limits.

Use them together — Well-Architected for workload-level reviews on a cadence, Trusted Advisor for account-level continuous monitoring.

## AWS Well-Architected Tool

The Well-Architected Tool lives in the console. You define a **workload** (name, regions, lens), pick lenses (the **AWS Well-Architected Framework** lens by default; specialised lenses for **Serverless**, **SaaS**, **Foundational Technical Review** (FTR), **Machine Learning**, **IoT**, **Data Analytics**, **Hybrid Networking**), and walk through the questionnaire.

Each question has multiple options corresponding to best practices. Unchecked options become **HRIs (High Risk Issues)** or **MRIs (Medium Risk Issues)**, scored per pillar. You add improvement notes, link to JIRA tickets, and re-run reviews every 6–12 months to see how the workload's posture is trending.

**Custom lenses** let you encode organisation-specific standards on top of the AWS baseline — useful for regulated industries where your compliance team has its own list.

The tool is free; the value is the structured conversation it forces. Most teams find HRIs they were aware of but had never named, and a few they hadn't considered. The first review takes ~4 hours; subsequent ones are an hour or two.

## Trusted Advisor

Trusted Advisor is the always-on companion. It runs checks across five categories — **Cost Optimization**, **Performance**, **Security**, **Fault Tolerance**, **Service Limits** — and produces a coloured dashboard with concrete findings.

The basic tier (all accounts get it free) includes seven core checks — chiefly the security ones (root MFA, public S3 buckets, exposed access keys, IAM use) and Service Limits. **Business** and **Enterprise** support tiers unlock the full set: hundreds of checks across all categories, including idle resources, underutilised instances, low Reserved Instance utilisation, and missing redundancy.

Trusted Advisor can publish to EventBridge — "alert me when a new public S3 bucket appears" is a one-rule wire-up. The standard pattern: weekly review of new findings, with cost recommendations going to the FinOps queue and security findings to security.

The full enterprise feature set is now also exposed via **Security Hub** (centralised findings across GuardDuty, Inspector, Macie, IAM Access Analyzer, and Trusted Advisor security checks) and **AWS Config** (continuous compliance evaluation). For organisations with many accounts, route Trusted Advisor and Security Hub findings to a centralised security account.

# Act 3 — SAA exam — blueprint, reading scenarios, study plan

The **AWS Certified Solutions Architect — Associate (SAA-C03)** exam tests architectural competence across the platform. The exam blueprint maps cleanly onto the thirteen notebooks before this one; this act walks the mapping, then covers how to read scenario questions and the distractor patterns that show up repeatedly.

If the foundations are solid, the exam is mostly translation: recognise which AWS noun the question is describing, eliminate options that cannot satisfy the constraints, decide between survivors on the explicit preference.

## SAA blueprint — where each notebook fits

The SAA-C03 exam has four domains:

| Domain | Weight | Notebooks |
|---|---|---|
| Design Secure Architectures | 30% | **02** (IAM, Organizations), **11** (KMS, WAF, Shield, GuardDuty), **06** (VPC, SGs, NACLs) |
| Design Resilient Architectures | 26% | **03** (ELB, ASG), **08** (RDS Multi-AZ, Aurora), **13** (DR strategies, Backup) |
| Design High-Performing Architectures | 24% | **03** (compute), **04** (serverless), **05** (storage), **07** (CloudFront), **08–10** (data and integration) |
| Design Cost-Optimized Architectures | 20% | **13** (RIs, Savings Plans, Spot, S3 tiers), **05** (S3 classes), **03** (instance choice) |

Notebooks **01** (foundations) and **12** (observability + governance) cut across all four domains; notebook **14** (this one) is the integration. There is no SAA topic that is not covered somewhere in the previous thirteen.

The domains are weighted roughly equally; don't over-study one. The hardest tend to be **Resilient Architectures** (multi-Region, RDS replicas vs Multi-AZ vs Aurora, Route 53 routing) and **Secure Architectures** (encryption at rest with KMS, S3 bucket policies, VPC endpoints) — both for the same reason: each has several services that look similar and a question often hinges on a small distinguishing constraint.

## Reading a scenario question

SAA questions are paragraphs, not one-liners. A repeatable reading pattern turns time into accuracy:

1. **Find the verb first.** "You need to **ensure** / **provide** / **minimise** / **prevent** ..." — that is the goal. Skim past the setup until you've found it.
2. **Highlight the constraints.** *Least operational overhead* / *most cost-effective* / *minimum downtime* / *no public IP* / *across two Regions* — each constraint eliminates options.
3. **Match the constraint to a service, not to a feature.** "Global, edge-cached, with WAF" → CloudFront, not ALB. "Reliable message processing in order" → SQS FIFO, not SQS Standard or SNS.
4. **Eliminate by capability mismatch.** If an option simply cannot do what's asked (Classic Load Balancer has no host-based routing; DynamoDB doesn't support SQL JOINs), strike it. Usually two of four eliminate cleanly from capability alone.
5. **Decide the survivors on the explicit preference.** *Least operational overhead* → managed service, not EC2-hosted. *Cost-effective* → S3 over EFS where it fits. *Minimum downtime* → blue/green or canary, not in-place. *Multi-Region active-active* → Aurora Global Database + DynamoDB Global Tables + Route 53 latency / failover, not Multi-AZ alone.

Most SAA scenarios reward recognising *which AWS noun is being described*. If the right noun is already in your head — and the previous notebooks put it there — the question becomes translation rather than guess.

## Distractor patterns to recognise

A few recurring traps that show up across the question pool:

- **"Customer wants minimum operational overhead"** — pick the managed service (Lambda over EC2, Fargate over self-managed ECS, Aurora Serverless over self-managed Postgres on EC2).
- **"Encrypt data at rest with a customer-managed key"** — KMS CMK, not AWS-managed default keys.
- **"Most cost-effective" + "infrequent access"** — usually S3 Standard-IA or Glacier; check the *first-byte latency* tolerance to pick which.
- **CLB / NAT instance / EC2-Classic** — usually wrong answers in modern questions; ALB / NAT Gateway / VPC are the modern primitives.
- **Multi-AZ vs Read Replica** — Multi-AZ is for failover, *not* for read scaling. Read replicas are for read scaling, *not* for automatic failover (until you promote one). Questions that say "increase read throughput" want read replicas.
- **SQS Standard vs SQS FIFO** — FIFO when the question says "exactly once" or "in order"; Standard otherwise.
- **SNS vs SQS vs EventBridge** — work queue → SQS; push notification fan-out → SNS; event bus with routing rules → EventBridge.
- **Security Group vs NACL** — SG is stateful + instance-level; NACL is stateless + subnet-level. "Block one IP at subnet level" → NACL.
- **VPC Endpoint Gateway vs Interface** — S3 and DynamoDB get Gateway endpoints (free); everything else uses Interface endpoints (per-hour cost).
- **CloudFront vs Global Accelerator** — HTTP/S caching at the edge → CloudFront; non-HTTP global TCP/UDP routing → Global Accelerator.
- **Route 53 routing policies** — match the verb: failover → Failover, latency → Latency, geographic → Geolocation, weighted → Weighted, location-specific endpoint → Geoproximity.
- **EFS vs FSx vs EBS** — multi-AZ shared POSIX → EFS; Windows shares → FSx for Windows; HPC scratch → FSx for Lustre; single-instance block → EBS.
- **STS AssumeRole vs IAM user** — cross-account, federated, or temporary → AssumeRole. Long-lived access keys are almost always the wrong answer on new questions.
- **Aurora Global Database** vs **Cross-Region Read Replicas** — Global DB is the modern multi-Region answer (RPO ~1s, RTO < 1min); cross-Region replicas are the older / weaker variant.
- **S3 Object Lock + Vault Lock** — when the question mentions "WORM" or "compliance" or "ransomware".
- **Lambda VPC config** — needed only when Lambda must reach a private resource; adds cold-start cost.

When a question offers a tempting old-school answer alongside a newer managed primitive, the newer one is almost always correct.

## Study plan

A practical four-week run at SAA-C03 once you have read this series:

- **Week 1 — Foundations, IAM, VPC, Compute.** Re-read notebooks 01, 02, 03, 06. Build a lab: a VPC with public + private subnets, an ASG behind an ALB across two AZs, an IAM role with least-privilege policies. Practise reading IAM policy JSON.
- **Week 2 — Serverless, Storage, DNS, CDN.** Notebooks 04, 05, 07. Deploy a Lambda + API Gateway pair. Stand up an S3 bucket with versioning, lifecycle, and a CloudFront distribution. Configure Route 53 with a failover routing policy.
- **Week 3 — Data, Integration, Security.** Notebooks 08, 09, 10, 11. Build an RDS Multi-AZ instance; create a DynamoDB table with GSIs; set up an SQS-to-Lambda pipeline; encrypt an EBS volume with a KMS CMK.
- **Week 4 — Observability, Resilience, Cost.** Notebooks 12, 13. Run an AWS Backup plan with cross-Region copy. Use Cost Explorer to identify a savings opportunity. Take an official practice exam — review every wrong answer against the source notebook.

The last step is the most important. The official AWS practice questions are calibrated to the exam difficulty; the third-party question banks vary widely. Treat each wrong answer as an instruction to re-read a notebook section, not as a flashcard to memorise.

On exam day: read the verb first, eliminate Classic-era and unmanaged options, prefer managed services and modern primitives, and trust the picking rules for the confusable triplets (SQS/SNS/EventBridge, ALB/NLB/GLB, Multi-AZ/Read Replica/Aurora Global).

## Closing

Fourteen notebooks in, the shape of AWS should be steady in your head: an account is a security and billing boundary, Regions and AZs are the physical substrate, IAM enforces identity at the chokepoint, a portfolio of compute / storage / network / data / integration / security services each solves a specific problem at a specific layer, and the operational disciplines — observability, deployment, backup, DR, cost — turn the parts into a running system.

AWS is wide but it composes. Every workload you will design from here uses the same primitives in different combinations. The work is no longer learning what the pieces are; it is recognising which combination fits the requirement in front of you. The Well-Architected Framework is the formal vocabulary for those trade-offs; the SAA exam is the calibrated test of whether you can name the right primitive on demand.

Good luck with the exam, and more importantly, with the systems you go on to build.